# Pandas: Basics

*Pandas is a Python library for creating and manipulating DataFrames — two-dimensional objects designed to store data.*

Key capabilities:
- High-performance manipulation of text, integers, numbers, and dates
- Data alignment, reshaping, and pivoting
- Intelligent slicing, grouping, and subsetting
- Merging and joining datasets

Resources: [Official Pandas Docs](https://pandas.pydata.org/about/) · [W3Schools Pandas](https://www.w3schools.com/python/pandas/default.asp) · [Pandas Cheat Sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)

| | Contents |
|-|----------|
| 1. | [Introduction to DataFrames](#1-introduction-to-dataframes) |
| 2. | [Working with Rows](#2-working-with-rows) |
| 3. | [Working with Columns](#3-working-with-columns) |
| 4. | [Filtering Data](#4-filtering-data) |
| 5. | [Sort and Count](#5-sort-and-count) |
| 6. | [Combining DataFrames](#6-combining-dataframes) |
| 7. | [Making the Most of Pandas](#7-making-the-most-of-pandas) |

## 1. Introduction to DataFrames

Pandas **DataFrames** are the basic unit upon which all operations take place. Think of them as spreadsheets: rows of records and named columns.

- Pandas can import data from many sources — **CSV** files, **Excel** spreadsheets, and also **JSON** (JavaScript Object Notation), a format that looks a lot like a Python list of dictionaries. These can be loaded locally (from your computer or Jupyter Hub space) or remotely (from a URL).

- Real-world JSON often contains **nested objects** (a value that is itself a dict) or **arrays** (a value that is a list). We need to **normalize** that structure into a flat table before we can work with it in Pandas.


### Import Libraries

`requests` fetches data from a URL; `pandas` turns it into a DataFrame.

In [1]:
import requests
import pandas as pd

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.1) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


### Meet the Melbourne Concert Programs Dataset

We will work with data drawn from the **Melbourne Rare Concert Programs** project — a collection of historic concert programs (Melbourne Liedertafel, Musashino Academia Musicae, Musical Society of Victoria, and others) that have been digitized and parsed into structured records.

Each concert program has been broken into many small **item records**, one per fact extracted from the page. A `record_type` field tells you what kind of fact each row describes:

| `record_type` | What it captures | Extra fields |
|---|---|---|
| `venue` | Where the concert was held | `place` |
| `date` | When the concert took place | `date_text` |
| `organization` | A presenting or sponsoring body | `name`, `role` |
| `patron` | A named patron of the event | `name`, `role` |
| `work` | A piece of music on the program | `composer`, `title`, `movement_or_selection` |
| `performer` | A performer or role in the concert | `name`, `part_or_instrument`, `associated_work` |

Every record also carries shared metadata about the source document: `filename`, `manifest_contents`, `manifest_date`, `manifest_organization`, `page_number`, `source_pdf`, `source_text`, `txt_source_file`, and `review_flags` (a list of notes left by the extraction pipeline, e.g. *"Composer inferred from preceding line"*).

Because different `record_type`s carry different fields, a single record looks like this:

```json
{
  "record_type":           "work",
  "composer":              "Heinrich Hofmann",
  "title":                 "Harald's Bridal Voyage",
  "movement_or_selection": null,
  "manifest_organization": "Melbourne Liedertafel",
  "manifest_date":         "1889",
  "page_number":           7,
  "filename":              "UDC20260028-13.pdf",
  "review_flags":          []
}
```

There are no nested dicts to flatten here, but `pd.json_normalize()` is still the right tool: it turns the list of (differently-shaped) dicts into one flat table, filling in `NaN` wherever a field doesn't apply to a given `record_type`.

### Load and Normalize the JSON

In [2]:
url = 'https://raw.githubusercontent.com/RichardFreedman/Summer2026/e9e8a4a716f11d12a36a9c7db63c87503b1444aa/Melbourne_Rare_Concert/Structured%20Data/concert_program_items.json'
response = requests.get(url)
raw_data = response.json()   # a Python list of dicts

# Flatten into a DataFrame (safe even when there's nothing nested to flatten)
concerts = pd.json_normalize(raw_data)

concerts.head(5)

,record_type,filename,source_pdf,txt_source_file,manifest_contents,manifest_date,manifest_organization,page_number,source_text,review_flags,place,date_text,name,role,composer,title,movement_or_selection,part_or_instrument,associated_work
0,venue,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,"Town Hall, Melbourne",[],"Town Hall, Melbourne",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,venue,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,20,"TOWN HALL,\nMELBOURNE",[],"TOWN HALL, MELBOURNE",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,date,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,"Wednesday Evening, Sixth November, 1889",[],NaN,"Wednesday Evening, Sixth November, 1889",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,date,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,20,"WEDNESDAY EVENING\n6TH NOVEMBER, 1889.",[],NaN,"WEDNESDAY EVENING 6TH NOVEMBER, 1889.",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,Metropolitan Liedertafel,[],NaN,NaN,Metropolitan Liedertafel,presenting organization,NaN,NaN,NaN,NaN,NaN


### Inspect the DataFrame

A few essential methods for getting to know a new DataFrame:

| Method / Attribute | What it shows |
|--------------------|---------------|
| `df.head(n)` | First `n` rows (default 5) |
| `df.tail(n)` | Last `n` rows (default 5) |
| `df.info()` | Column names, non-null counts, data types |
| `df.shape` | `(rows, columns)` — note: no parentheses |
| `df.describe()` | Basic statistics for numeric columns |

In [3]:
concerts.head(10)

,record_type,filename,source_pdf,txt_source_file,manifest_contents,manifest_date,manifest_organization,page_number,source_text,review_flags,place,date_text,name,role,composer,title,movement_or_selection,part_or_instrument,associated_work
0,venue,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,"Town Hall, Melbourne",[],"Town Hall, Melbourne",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,venue,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,20,"TOWN HALL,\nMELBOURNE",[],"TOWN HALL, MELBOURNE",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,date,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,"Wednesday Evening, Sixth November, 1889",[],NaN,"Wednesday Evening, Sixth November, 1889",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,date,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,20,"WEDNESDAY EVENING\n6TH NOVEMBER, 1889.",[],NaN,"WEDNESDAY EVENING 6TH NOVEMBER, 1889.",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,Metropolitan Liedertafel,[],NaN,NaN,Metropolitan Liedertafel,presenting organization,NaN,NaN,NaN,NaN,NaN
5,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,7,Liedertafel,[Appears to be abbreviated form of 'Metropolit...,NaN,NaN,Liedertafel,choir,NaN,NaN,NaN,NaN,NaN
6,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,4,Victorian Orchestra,[],NaN,NaN,Victorian Orchestra,orchestra,NaN,NaN,NaN,NaN,NaN
7,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,4,A Selected Choir of Boys' Voices,[],NaN,NaN,A Selected Choir of Boys' Voices,choir,NaN,NaN,NaN,NaN,NaN
8,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,13,"Boys' Choir, Victorian Orchestra\nAND Liedertafel",[“Boys' Choir” likely refers to the “A Selecte...,NaN,NaN,Boys' Choir,choir,NaN,NaN,NaN,NaN,NaN
9,patron,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,5,PATRON\nHIS EXCELLENCY SIR HENRY BROUGHAM LOCH...,[],NaN,NaN,"HIS EXCELLENCY SIR HENRY BROUGHAM LOCH, K.C.B.",Patron,NaN,NaN,NaN,NaN,NaN


In [4]:
concerts.info()

<class 'pandas.DataFrame'>
RangeIndex: 11379 entries, 0 to 11378
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   record_type            11379 non-null  str   
 1   filename               11379 non-null  str   
 2   source_pdf             11379 non-null  str   
 3   txt_source_file        11379 non-null  str   
 4   manifest_contents      11379 non-null  str   
 5   manifest_date          11379 non-null  str   
 6   manifest_organization  11379 non-null  str   
 7   page_number            11379 non-null  int64 
 8   source_text            11379 non-null  str   
 9   review_flags           11379 non-null  object
 10  place                  454 non-null    str   
 11  date_text              580 non-null    str   
 12  name                   7456 non-null   str   
 13  role                   820 non-null    str   
 14  composer               2772 non-null   str   
 15  title                  2889 no

In [5]:
concerts.shape

(11379, 19)

In [6]:
concerts.describe()

,page_number
count,11379.000000
mean,8.751208
std,6.745744
min,1.000000
25%,5.000000
50%,6.000000
75%,11.000000
max,75.000000


## 2. Working with Rows

By default Pandas shows only the first and last five rows. Here are your options:

| Method | What it does |
|--------|-------------|
| `df.head(n)` | First `n` rows |
| `df.tail(n)` | Last `n` rows |
| `df.sample(n)` | Random sample of `n` rows |
| `pd.set_option('display.max_rows', None)` | Show all rows |

In [7]:
concerts.head(5)
# concerts.tail(5)
# concerts.sample(10)

,record_type,filename,source_pdf,txt_source_file,manifest_contents,manifest_date,manifest_organization,page_number,source_text,review_flags,place,date_text,name,role,composer,title,movement_or_selection,part_or_instrument,associated_work
0,venue,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,"Town Hall, Melbourne",[],"Town Hall, Melbourne",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,venue,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,20,"TOWN HALL,\nMELBOURNE",[],"TOWN HALL, MELBOURNE",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,date,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,"Wednesday Evening, Sixth November, 1889",[],NaN,"Wednesday Evening, Sixth November, 1889",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,date,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,20,"WEDNESDAY EVENING\n6TH NOVEMBER, 1889.",[],NaN,"WEDNESDAY EVENING 6TH NOVEMBER, 1889.",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,Metropolitan Liedertafel,[],NaN,NaN,Metropolitan Liedertafel,presenting organization,NaN,NaN,NaN,NaN,NaN


### Selecting Rows: `iloc` and `loc`

#### `iloc` — select by **integer index position**

Syntax: `df.iloc[start_row : end_row, start_col : end_col]`

- The range is *inclusive* at the start and *exclusive* at the end: `iloc[0:5]` gives rows 0–4.
- Omit either end to go from the beginning or to the end: `iloc[:5]` = first 5 rows.
- Omit the column range to get all columns.
- Negative indices count from the end: `-1` is the last column.

#### `loc` — select by **label**

Most useful when selecting columns by name, or when the index is a string.

In [8]:
# Rows 5-9, all columns
concerts.iloc[5:10]

,record_type,filename,source_pdf,txt_source_file,manifest_contents,manifest_date,manifest_organization,page_number,source_text,review_flags,place,date_text,name,role,composer,title,movement_or_selection,part_or_instrument,associated_work
5,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,7,Liedertafel,[Appears to be abbreviated form of 'Metropolit...,NaN,NaN,Liedertafel,choir,NaN,NaN,NaN,NaN,NaN
6,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,4,Victorian Orchestra,[],NaN,NaN,Victorian Orchestra,orchestra,NaN,NaN,NaN,NaN,NaN
7,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,4,A Selected Choir of Boys' Voices,[],NaN,NaN,A Selected Choir of Boys' Voices,choir,NaN,NaN,NaN,NaN,NaN
8,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,13,"Boys' Choir, Victorian Orchestra\nAND Liedertafel",[“Boys' Choir” likely refers to the “A Selecte...,NaN,NaN,Boys' Choir,choir,NaN,NaN,NaN,NaN,NaN
9,patron,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,5,PATRON\nHIS EXCELLENCY SIR HENRY BROUGHAM LOCH...,[],NaN,NaN,"HIS EXCELLENCY SIR HENRY BROUGHAM LOCH, K.C.B.",Patron,NaN,NaN,NaN,NaN,NaN


In [9]:
# All rows, just the record-type and title-ish columns
concerts.iloc[:, [concerts.columns.get_loc('record_type'), concerts.columns.get_loc('source_text')]]

,record_type,source_text
0,venue,"Town Hall, Melbourne"
1,venue,"TOWN HALL,\nMELBOURNE"
2,date,"Wednesday Evening, Sixth November, 1889"
3,date,"WEDNESDAY EVENING\n6TH NOVEMBER, 1889."
4,organization,Metropolitan Liedertafel
...,...,...
11374,performer,Mr. Fries
11375,performer,Contra Bass Tuba—\n Mr. Lüttich
11376,performer,Kettle Drums—\n Mr. Clay
11377,performer,Bass Drum—\n Mr. Corbett


In [10]:
# Rows 0-4, last column only
concerts.iloc[0:5, -1]

0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
Name: associated_work, dtype: str

### Dropping Rows

Remove rows by index number with `.drop()`, then restore a clean 0-based index with `.reset_index(drop=True)`.

In [11]:
concerts_trimmed = concerts.drop([0, 1])
concerts_trimmed = concerts_trimmed.reset_index(drop=True)
concerts_trimmed.head(5)

,record_type,filename,source_pdf,txt_source_file,manifest_contents,manifest_date,manifest_organization,page_number,source_text,review_flags,place,date_text,name,role,composer,title,movement_or_selection,part_or_instrument,associated_work
0,date,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,"Wednesday Evening, Sixth November, 1889",[],NaN,"Wednesday Evening, Sixth November, 1889",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,date,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,20,"WEDNESDAY EVENING\n6TH NOVEMBER, 1889.",[],NaN,"WEDNESDAY EVENING 6TH NOVEMBER, 1889.",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,Metropolitan Liedertafel,[],NaN,NaN,Metropolitan Liedertafel,presenting organization,NaN,NaN,NaN,NaN,NaN
3,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,7,Liedertafel,[Appears to be abbreviated form of 'Metropolit...,NaN,NaN,Liedertafel,choir,NaN,NaN,NaN,NaN,NaN
4,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,4,Victorian Orchestra,[],NaN,NaN,Victorian Orchestra,orchestra,NaN,NaN,NaN,NaN,NaN


## 3. Working with Columns

| # | Task |
|---|------|
| 1 | Show all column names |
| 2 | Add a column |
| 3 | Drop a column |
| 4 | Rename a column (or columns) |
| 5 | Show column data types |
| 6 | Reorder or subset columns |

### Show All Column Names

Note the absence of `()` — `.columns` is an attribute, not a method.

In [12]:
# As an Index object
concerts.columns

Index(['record_type', 'filename', 'source_pdf', 'txt_source_file',
       'manifest_contents', 'manifest_date', 'manifest_organization',
       'page_number', 'source_text', 'review_flags', 'place', 'date_text',
       'name', 'role', 'composer', 'title', 'movement_or_selection',
       'part_or_instrument', 'associated_work'],
      dtype='str')

In [13]:
# Sorted alphabetically
concerts.columns.sort_values()

Index(['associated_work', 'composer', 'date_text', 'filename',
       'manifest_contents', 'manifest_date', 'manifest_organization',
       'movement_or_selection', 'name', 'page_number', 'part_or_instrument',
       'place', 'record_type', 'review_flags', 'role', 'source_pdf',
       'source_text', 'title', 'txt_source_file'],
      dtype='str')

### Add a Column

Assign a new column name to an expression evaluated row-by-row.
Here we flag any record that has at least one entry in `review_flags` — meaning the extraction pipeline had to infer or normalize something on that row.

In [14]:
concerts['was_flagged'] = concerts['review_flags'].apply(lambda flags: len(flags) > 0)
concerts[['record_type', 'source_text', 'was_flagged']].head(10)

,record_type,source_text,was_flagged
0,venue,"Town Hall, Melbourne",False
1,venue,"TOWN HALL,\nMELBOURNE",False
2,date,"Wednesday Evening, Sixth November, 1889",False
3,date,"WEDNESDAY EVENING\n6TH NOVEMBER, 1889.",False
4,organization,Metropolitan Liedertafel,False
5,organization,Liedertafel,True
6,organization,Victorian Orchestra,False
7,organization,A Selected Choir of Boys' Voices,False
8,organization,"Boys' Choir, Victorian Orchestra\nAND Liedertafel",True
9,patron,PATRON\nHIS EXCELLENCY SIR HENRY BROUGHAM LOCH...,False


### Drop a Column

Columns to drop must be passed as a **list**, even if there is only one.

In [15]:
concerts = concerts.drop(columns=['was_flagged'])
concerts.columns

Index(['record_type', 'filename', 'source_pdf', 'txt_source_file',
       'manifest_contents', 'manifest_date', 'manifest_organization',
       'page_number', 'source_text', 'review_flags', 'place', 'date_text',
       'name', 'role', 'composer', 'title', 'movement_or_selection',
       'part_or_instrument', 'associated_work'],
      dtype='str')

### Rename Columns

**Option 1** — copy-then-drop (useful for a single column):

In [16]:
# Rename 'manifest_organization' to 'presenting_org'
concerts['presenting_org'] = concerts['manifest_organization']
concerts = concerts.drop(columns=['manifest_organization'])
concerts.columns

Index(['record_type', 'filename', 'source_pdf', 'txt_source_file',
       'manifest_contents', 'manifest_date', 'page_number', 'source_text',
       'review_flags', 'place', 'date_text', 'name', 'role', 'composer',
       'title', 'movement_or_selection', 'part_or_instrument',
       'associated_work', 'presenting_org'],
      dtype='str')

**Option 2** — rename via a dictionary (cleaner for multiple columns at once):

In [17]:
col_dict = {
    'presenting_org': 'manifest_organization',  # rename back for tidiness
    'page_number':    'page'
}

concerts = concerts.rename(columns=col_dict)
concerts.columns

Index(['record_type', 'filename', 'source_pdf', 'txt_source_file',
       'manifest_contents', 'manifest_date', 'page', 'source_text',
       'review_flags', 'place', 'date_text', 'name', 'role', 'composer',
       'title', 'movement_or_selection', 'part_or_instrument',
       'associated_work', 'manifest_organization'],
      dtype='str')

**Tip: `dict.fromkeys()` as a shortcut**

Generate a skeleton dictionary with your current column names as keys. Fill in only the ones you want to rename — handy when there are many columns.

In [18]:
col_dict = dict.fromkeys(concerts.columns)
col_dict

{'record_type': None,
 'filename': None,
 'source_pdf': None,
 'txt_source_file': None,
 'manifest_contents': None,
 'manifest_date': None,
 'page': None,
 'source_text': None,
 'review_flags': None,
 'place': None,
 'date_text': None,
 'name': None,
 'role': None,
 'composer': None,
 'title': None,
 'movement_or_selection': None,
 'part_or_instrument': None,
 'associated_work': None,
 'manifest_organization': None}

### Show Column Data Types

Each column has a **dtype** controlling which operations are valid.
You cannot apply string methods to a numeric column, or math to a string column.

In [19]:
concerts.dtypes

record_type                 str
filename                    str
source_pdf                  str
txt_source_file             str
manifest_contents           str
manifest_date               str
page                      int64
source_text                 str
review_flags             object
place                       str
date_text                   str
name                        str
role                        str
composer                    str
title                       str
movement_or_selection       str
part_or_instrument          str
associated_work             str
manifest_organization       str
dtype: object

### Reorder Columns / Create a Subset

Pass a list of column names in the desired order. Omit any column to exclude it — this is how you create a **subset DataFrame**.

In [20]:
column_list = ['record_type', 'manifest_organization', 'manifest_date', 'page', 'name', 'composer', 'title', 'source_text']
concerts_short = concerts[column_list]
concerts_short.head(10)

,record_type,manifest_organization,manifest_date,page,name,composer,title,source_text
0,venue,Melbourne Liedertafel,1889,3,NaN,NaN,NaN,"Town Hall, Melbourne"
1,venue,Melbourne Liedertafel,1889,20,NaN,NaN,NaN,"TOWN HALL,\nMELBOURNE"
2,date,Melbourne Liedertafel,1889,3,NaN,NaN,NaN,"Wednesday Evening, Sixth November, 1889"
3,date,Melbourne Liedertafel,1889,20,NaN,NaN,NaN,"WEDNESDAY EVENING\n6TH NOVEMBER, 1889."
4,organization,Melbourne Liedertafel,1889,3,Metropolitan Liedertafel,NaN,NaN,Metropolitan Liedertafel
5,organization,Melbourne Liedertafel,1889,7,Liedertafel,NaN,NaN,Liedertafel
6,organization,Melbourne Liedertafel,1889,4,Victorian Orchestra,NaN,NaN,Victorian Orchestra
7,organization,Melbourne Liedertafel,1889,4,A Selected Choir of Boys' Voices,NaN,NaN,A Selected Choir of Boys' Voices
8,organization,Melbourne Liedertafel,1889,13,Boys' Choir,NaN,NaN,"Boys' Choir, Victorian Orchestra\nAND Liedertafel"
9,patron,Melbourne Liedertafel,1889,5,"HIS EXCELLENCY SIR HENRY BROUGHAM LOCH, K.C.B.",NaN,NaN,PATRON\nHIS EXCELLENCY SIR HENRY BROUGHAM LOCH...


### A Column is a Series

An individual column is a **Series** — a labelled one-dimensional array with its own methods.

| Expression | What it returns |
|------------|----------------|
| `df["col"]` | The column as a Series |
| `df["col"].unique()` | Array of unique values |
| `df["col"].nunique()` | Count of unique values |
| `df["col"].value_counts()` | Frequency of each value |

In [21]:
print("Record types:", concerts["record_type"].unique())
print("\nDistinct presenting organizations:", concerts["manifest_organization"].nunique())

Record types: <ArrowStringArray>
['venue', 'date', 'organization', 'patron', 'work', 'performer']
Length: 6, dtype: str

Distinct presenting organizations: 17


## 4. Filtering Data

A **boolean mask** — a Series of `True`/`False` values — lets you keep only the rows that match a condition. Put the condition inside `df[ ... ]`.

| Expression | What it does |
|---|---|
| `df[df['col'] == value]` | Rows where `col` equals `value` |
| `df[df['col'].isin([a, b])]` | Rows where `col` is one of several values |
| `df[(cond1) & (cond2)]` | Rows matching **both** conditions |
| `df[(cond1) \| (cond2)]` | Rows matching **either** condition |
| `df['col'].notna()` | Rows where `col` is not missing |

Note the parentheses around each condition — they're required when combining conditions with `&` / `|`.

In [22]:
# Just the "work" records -- the pieces of music performed
works = concerts_short[concerts_short['record_type'] == 'work']
works.head(10)

,record_type,manifest_organization,manifest_date,page,name,composer,title,source_text
10,work,Melbourne Liedertafel,1889,7,NaN,Heinrich Hofmann,Harald's Bridal Voyage,"Cantata ""Harald's Bridal Voyage""\nHeinrich Hof..."
11,work,Melbourne Liedertafel,1889,11,NaN,Beethoven,Concerto in E Flat Major,Pianoforte Solo: Concerto in E Flat Major Beet...
12,work,Melbourne Liedertafel,1889,12,NaN,NaN,Serenade,Serenade\n\n(TENOR SOLO Mr. W. Clarence Fraser)
13,work,Melbourne Liedertafel,1889,12,NaN,NaN,L'Estasi,"Song\n\n""L'Estasi""\n\nMiss O'Shannessy"
14,work,Melbourne Liedertafel,1889,13,NaN,Berlioz,La Damnation de Faust,"Selections from ""La Damnation de Faust"" \nBer..."
15,work,Melbourne Liedertafel,1889,13,NaN,Berlioz,La Damnation de Faust,2.—Easter Hymn
16,work,Melbourne Liedertafel,1889,13,NaN,Berlioz,La Damnation de Faust,3.—Danse des Sylphes
17,work,Melbourne Liedertafel,1889,13,NaN,Berlioz,La Damnation de Faust,4.—Chorus of Soldiers and Students
18,work,Melbourne Liedertafel,1889,14,NaN,Coombs,Bedouin Love Song,"Song ""Bedouin Love Song"" Coombs"
19,work,Melbourne Liedertafel,1889,14,NaN,Zinkel,Farewell,"Part Song ""Farewell"" Zinkel"


In [23]:
# Just the performers, for a single organization
liedertafel_performers = concerts_short[
    (concerts_short['record_type'] == 'performer') &
    (concerts_short['manifest_organization'] == 'Melbourne Liedertafel')
]
liedertafel_performers.head(10)

,record_type,manifest_organization,manifest_date,page,name,composer,title,source_text
22,performer,Melbourne Liedertafel,1889,3,Mr. Julius Herz,NaN,NaN,Conductor: Mr. Julius Herz
23,performer,Melbourne Liedertafel,1889,4,Mr. G. B. Fentum,NaN,NaN,"Mr. G. B. Fentum, Hon. Pianist"
24,performer,Melbourne Liedertafel,1889,12,Miss O'Shannessy,NaN,NaN,"""L'Estasi""\n\nMiss O'Shannessy"
25,performer,Melbourne Liedertafel,1889,14,Mr. A. H. Gee,NaN,NaN,"Song ""Bedouin Love Song"" Coombs\nMr. A. ..."
26,performer,Melbourne Liedertafel,1889,7,Mr. A. H. Gee,NaN,NaN,Solos by Mr. A. H. Gee and Mr. Henry Rofe
27,performer,Melbourne Liedertafel,1889,7,Mr. Henry Rofe,NaN,NaN,Solos by Mr. A. H. Gee and Mr. Henry Rofe
28,performer,Melbourne Liedertafel,1889,11,Miss Florence Menk-Meyer,NaN,NaN,Pianoforte Solo: Concerto in E Flat Major Beet...
29,performer,Melbourne Liedertafel,1889,12,Mr. W. Clarence Fraser,NaN,NaN,(TENOR SOLO Mr. W. Clarence Fraser)
30,performer,Melbourne Liedertafel,1889,6,A. Cameron,NaN,NaN,A. Cameron
31,performer,Melbourne Liedertafel,1889,6,R. Jones,NaN,NaN,R. Jones


In [24]:
# Works or performers, across two organizations, using .isin()
subset = concerts_short[
    concerts_short['record_type'].isin(['work', 'performer']) &
    concerts_short['manifest_organization'].isin(['Melbourne Liedertafel', 'Musical Society of Victoria'])
]
subset['manifest_organization'].value_counts()

manifest_organization
Melbourne Liedertafel          5884
Musical Society of Victoria     486
Name: count, dtype: int64

In [25]:
# Works with a named composer (drop rows where composer is missing)
named_composer_works = works[works['composer'].notna()]
named_composer_works[['manifest_organization', 'composer', 'title']].head(10)

,manifest_organization,composer,title
10,Melbourne Liedertafel,Heinrich Hofmann,Harald's Bridal Voyage
11,Melbourne Liedertafel,Beethoven,Concerto in E Flat Major
14,Melbourne Liedertafel,Berlioz,La Damnation de Faust
15,Melbourne Liedertafel,Berlioz,La Damnation de Faust
16,Melbourne Liedertafel,Berlioz,La Damnation de Faust
17,Melbourne Liedertafel,Berlioz,La Damnation de Faust
18,Melbourne Liedertafel,Coombs,Bedouin Love Song
19,Melbourne Liedertafel,Zinkel,Farewell
20,Melbourne Liedertafel,Strauss,"Vocal Waltz ""Blue Danube"""
149,Melbourne Liedertafel,RICHARD WAGNER,PARSIFAL


## 5. Sort and Count

Pandas has built-in methods for sorting and summarizing data — no loops needed.

### Sort Values

`sort_values()` sorts by any column, ascending by default.

In [26]:
# Earliest programs first, by manifest_date
concerts_short.sort_values('manifest_date').head(10)

,record_type,manifest_organization,manifest_date,page,name,composer,title,source_text
0,venue,Melbourne Liedertafel,1889,3,NaN,NaN,NaN,"Town Hall, Melbourne"
1,venue,Melbourne Liedertafel,1889,20,NaN,NaN,NaN,"TOWN HALL,\nMELBOURNE"
2,date,Melbourne Liedertafel,1889,3,NaN,NaN,NaN,"Wednesday Evening, Sixth November, 1889"
3,date,Melbourne Liedertafel,1889,20,NaN,NaN,NaN,"WEDNESDAY EVENING\n6TH NOVEMBER, 1889."
4,organization,Melbourne Liedertafel,1889,3,Metropolitan Liedertafel,NaN,NaN,Metropolitan Liedertafel
5,organization,Melbourne Liedertafel,1889,7,Liedertafel,NaN,NaN,Liedertafel
6,organization,Melbourne Liedertafel,1889,4,Victorian Orchestra,NaN,NaN,Victorian Orchestra
7,organization,Melbourne Liedertafel,1889,4,A Selected Choir of Boys' Voices,NaN,NaN,A Selected Choir of Boys' Voices
8,organization,Melbourne Liedertafel,1889,13,Boys' Choir,NaN,NaN,"Boys' Choir, Victorian Orchestra\nAND Liedertafel"
9,patron,Melbourne Liedertafel,1889,5,"HIS EXCELLENCY SIR HENRY BROUGHAM LOCH, K.C.B.",NaN,NaN,PATRON\nHIS EXCELLENCY SIR HENRY BROUGHAM LOCH...


In [27]:
# Latest page numbers first, within the 'work' records
works.sort_values('page', ascending=False).head(10)

,record_type,manifest_organization,manifest_date,page,name,composer,title,source_text
2532,work,Musashino Academia Musicae,1979,44,NaN,ベートーヴェン,第9シンフォニー,…ベートーヴェン没後150年祭の公式行事に参加…日・独・英の若人達の共演による「第9シンフォ...
2533,work,Musashino Academia Musicae,1979,42,NaN,モーツァルト,コシ・ファン・トゥッテ,従来毎年発表を行っているが、昭和53年度にはモーツァルト“コシ・ファン・トゥッテ”を上演した。
10971,work,London Royal Choral Society,1936,41,NaN,COLERIDGE-TAYLOR,HIAWATHA'S DEPARTURE,III.—HIAWATHA'S DEPARTURE
10868,work,London Royal Choral Society,1939,41,NaN,COLERIDGE-TAYLOR,HIAWATHA'S DEPARTURE,III.—HIAWATHA'S DEPARTURE
10867,work,London Royal Choral Society,1939,37,NaN,COLERIDGE-TAYLOR,THE DEATH OF MINNEHAHA,II.—THE DEATH OF MINNEHAHA
10970,work,London Royal Choral Society,1936,37,NaN,COLERIDGE-TAYLOR,THE DEATH OF MINNEHAHA,II.—THE DEATH OF MINNEHAHA
5031,work,Musashino Academia Musicae,1978,36,NaN,V. Novak,"7 Songs Op. 4/, Op. 16/, Op. 38/, Op. 74-1, 2/...",V. Novak (100th Anniversary of his Birth)\n\n7...
5032,work,Musashino Academia Musicae,1978,36,NaN,V. Novak,"Sonatina for Piano ""The Thief""",V. Novak (100th Anniversary of his Birth)\n\n7...
5028,work,Musashino Academia Musicae,1978,35,NaN,H. Distler,"Partita ""Wachet auf, ruft uns die Stimme""",H. Distler ......................................
5029,work,Musashino Academia Musicae,1978,35,NaN,J. Nepomuk David,"""Es ist ein Schnitter, heisst der Tod""",J. Nepomuk David ................................


### Count Values

`value_counts()` returns the frequency of each unique value in a column.

In [28]:
concerts['record_type'].value_counts()

record_type
performer       6636
work            2889
date             580
venue            454
organization     429
patron           391
Name: count, dtype: int64

In [29]:
concerts['manifest_organization'].value_counts()

manifest_organization
Melbourne Liedertafel                   6404
Musashino Academia Musicae              3468
Musical Society of Victoria              616
University of Michigan                   153
Asian Youth Music Camp                   130
London Royal Choral Society              117
London Symphony Orchestra                114
Australian Youth Orchestra                99
Pacific Contemporary Music Festival       71
Obergammerau Canada Educator's Visit      49
Paris Académie de musique et danse        43
Royal College of Music                    28
Royal Albert Hall                         27
Madang Amateur Theatrical Society         25
Lausanne Music Festival                   24
London Queens Hall Piano                   9
Hendersen Piano Lectures Glasgow           2
Name: count, dtype: int64

In [30]:
works['composer'].value_counts().head(15)

composer
J. Brahms          104
R. Schumann         87
W. A. Mozart        86
J. S. Bach          73
F. Schubert         72
F. Chopin           69
L. v. Beethoven     55
C. Debussy          55
H. Wolf             46
G. Verdi            35
G. Puccini          35
J. Haydn            31
F. Liszt            28
M. Ravel            26
R. Strauss          25
Name: count, dtype: int64

In [31]:
# Store a result as a new DataFrame
org_counts = pd.DataFrame(concerts['manifest_organization'].value_counts())
org_counts.head(10)

,count
manifest_organization,
Melbourne Liedertafel,6404
Musashino Academia Musicae,3468
Musical Society of Victoria,616
University of Michigan,153
Asian Youth Music Camp,130
London Royal Choral Society,117
London Symphony Orchestra,114
Australian Youth Orchestra,99
Pacific Contemporary Music Festival,71


## 6. Combining DataFrames

Even when working with a single source, you often split a DataFrame into subsets and then reassemble them. The two main tools are:

| Operation | When to use |
|-----------|-------------|
| **`pd.concat()`** | Stack DataFrames that share the same columns (add more rows) |
| **`pd.merge()`** | Join two DataFrames on a shared column (add more columns) |

### Concatenation

Here we split `concerts_short` into two `record_type`-based subsets, then stack them back into one frame.

In [32]:
performers = concerts_short[concerts_short['record_type'] == 'performer']

print(f"Works: {len(works)}  |  Performers: {len(performers)}")

combined = pd.concat([works, performers]).reset_index(drop=True)
print(f"Combined: {len(combined)}")
combined[['record_type', 'manifest_organization', 'composer', 'name']].head(10)

Works: 2889  |  Performers: 6636
Combined: 9525


,record_type,manifest_organization,composer,name
0,work,Melbourne Liedertafel,Heinrich Hofmann,NaN
1,work,Melbourne Liedertafel,Beethoven,NaN
2,work,Melbourne Liedertafel,NaN,NaN
3,work,Melbourne Liedertafel,NaN,NaN
4,work,Melbourne Liedertafel,Berlioz,NaN
5,work,Melbourne Liedertafel,Berlioz,NaN
6,work,Melbourne Liedertafel,Berlioz,NaN
7,work,Melbourne Liedertafel,Berlioz,NaN
8,work,Melbourne Liedertafel,Coombs,NaN
9,work,Melbourne Liedertafel,Zinkel,NaN


### Merging

`pd.merge()` joins two DataFrames that share a common column.

Key arguments:
- `left_on` / `right_on` — which column from each frame to match on
- `how="left"` — keep all rows from the left frame; fill with `NaN` where there's no match

Here we build a small lookup table mapping each presenting organization to its home city, then merge it into our main DataFrame.

In [33]:
# A small reference table
city_lookup = pd.DataFrame([
    {'manifest_organization': 'Melbourne Liedertafel',        'city': 'Melbourne'},
    {'manifest_organization': 'Musical Society of Victoria',  'city': 'Melbourne'},
    {'manifest_organization': 'Musashino Academia Musicae',   'city': 'Tokyo'},
    {'manifest_organization': 'London Royal Choral Society',  'city': 'London'},
    {'manifest_organization': 'London Symphony Orchestra',    'city': 'London'},
])

concerts_merged = pd.merge(
    left=concerts_short,
    right=city_lookup,
    on='manifest_organization',
    how='left'        # keep all concert rows; NaN where no match
)

concerts_merged.head(10)

,record_type,manifest_organization,manifest_date,page,name,composer,title,source_text,city
0,venue,Melbourne Liedertafel,1889,3,NaN,NaN,NaN,"Town Hall, Melbourne",Melbourne
1,venue,Melbourne Liedertafel,1889,20,NaN,NaN,NaN,"TOWN HALL,\nMELBOURNE",Melbourne
2,date,Melbourne Liedertafel,1889,3,NaN,NaN,NaN,"Wednesday Evening, Sixth November, 1889",Melbourne
3,date,Melbourne Liedertafel,1889,20,NaN,NaN,NaN,"WEDNESDAY EVENING\n6TH NOVEMBER, 1889.",Melbourne
4,organization,Melbourne Liedertafel,1889,3,Metropolitan Liedertafel,NaN,NaN,Metropolitan Liedertafel,Melbourne
5,organization,Melbourne Liedertafel,1889,7,Liedertafel,NaN,NaN,Liedertafel,Melbourne
6,organization,Melbourne Liedertafel,1889,4,Victorian Orchestra,NaN,NaN,Victorian Orchestra,Melbourne
7,organization,Melbourne Liedertafel,1889,4,A Selected Choir of Boys' Voices,NaN,NaN,A Selected Choir of Boys' Voices,Melbourne
8,organization,Melbourne Liedertafel,1889,13,Boys' Choir,NaN,NaN,"Boys' Choir, Victorian Orchestra\nAND Liedertafel",Melbourne
9,patron,Melbourne Liedertafel,1889,5,"HIS EXCELLENCY SIR HENRY BROUGHAM LOCH, K.C.B.",NaN,NaN,PATRON\nHIS EXCELLENCY SIR HENRY BROUGHAM LOCH...,Melbourne


### Cleaning Before Merging

A merge that returns fewer matches than expected usually means the key column is not formatted consistently between frames. Normalising to lowercase before merging is a simple first fix:

In [34]:
concerts_lower = concerts_short.copy()
lookup_lower = city_lookup.copy()

concerts_lower['manifest_organization'] = concerts_lower['manifest_organization'].str.lower()
lookup_lower['manifest_organization'] = lookup_lower['manifest_organization'].str.lower()

concerts_merged_clean = pd.merge(
    left=concerts_lower,
    right=lookup_lower,
    on='manifest_organization',
    how='left'
)

concerts_merged_clean.dropna(subset=['city'])[['manifest_organization', 'record_type', 'city']].head(10)

,manifest_organization,record_type,city
0,melbourne liedertafel,venue,Melbourne
1,melbourne liedertafel,venue,Melbourne
2,melbourne liedertafel,date,Melbourne
3,melbourne liedertafel,date,Melbourne
4,melbourne liedertafel,organization,Melbourne
5,melbourne liedertafel,organization,Melbourne
6,melbourne liedertafel,organization,Melbourne
7,melbourne liedertafel,organization,Melbourne
8,melbourne liedertafel,organization,Melbourne
9,melbourne liedertafel,patron,Melbourne


## 7. Making the Most of Pandas

Pandas is designed so that you almost **never need to write a `for` loop** — there is a built-in method for nearly every common operation on columns and DataFrames.

Before writing custom code, search the [documentation](https://pandas.pydata.org/about/) or the [cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf). The built-in path is almost always faster, more readable, and less error-prone.

**Key principle**: clean your data *before* analysing it. The lowercase-before-merge example above illustrates why — a small normalisation step can dramatically change your results.

Continue to: [Pandas: Clean Data](https://github.com/RichardFreedman/Encoding_Music/blob/main/01_Tutorials/05_Pandas_Clean_Data.md)